In [1]:
import pandas as pd
from pathlib import Path

project_folder = Path.cwd()

if project_folder.name == "notebooks":
    project_folder = project_folder.parent

data_folder = project_folder / "data"
feature_file = data_folder / "propertylens_featured.csv"

df = pd.read_csv(feature_file)

# Overall RERA comparison
rera_summary = (
    df.groupby("rera_approval")
    .agg(
        listings=("rera_approval", "size"),
        avg_price_lakh=("price_lakh", "mean"),
        median_price_lakh=("price_lakh", "median"),
        avg_rate_per_sqft=("rate_per_sqft", "mean"),
        median_rate_per_sqft=("rate_per_sqft", "median"),
        avg_area_sqft=("area_sqft", "mean"),
        median_area_sqft=("area_sqft", "median")
    )
    .reset_index()
)

print("RERA approval comparison:")
print(rera_summary.to_string(index=False))


# Property type distribution
rera_property_type = pd.crosstab(
    df["rera_approval"],
    df["flat_type"]
)

print("\nRERA approval by property type:")
print(rera_property_type.to_string())


# BHK distribution
rera_bhk = pd.crosstab(
    df["rera_approval"],
    df["bhk_group"]
)

print("\nRERA approval by BHK:")
print(rera_bhk.to_string())


# Status distribution
rera_status = pd.crosstab(
    df["rera_approval"],
    df["status"]
)

print("\nRERA approval by status:")
print(rera_status.to_string())


# Percentage of listings by property type within each RERA group
rera_property_type_pct = pd.crosstab(
    df["rera_approval"],
    df["flat_type"],
    normalize="index"
).mul(100).round(2)

print("\nProperty type share within each RERA group (%):")
print(rera_property_type_pct.to_string())


# Ready-to-move share within each RERA group
ready_to_move_rera = (
    df.assign(
        ready_to_move=df["status"].eq("Ready to move")
    )
    .groupby("rera_approval")["ready_to_move"]
    .mean()
    .mul(100)
    .round(2)
    .reset_index(name="ready_to_move_pct")
)

print("\nReady-to-move share:")
print(ready_to_move_rera.to_string(index=False))


# Observed difference between RERA groups
approved = rera_summary[
    rera_summary["rera_approval"] == "Approved by RERA"
].iloc[0]

not_approved = rera_summary[
    rera_summary["rera_approval"] == "Not approved by RERA"
].iloc[0]

median_price_difference = (
    approved["median_price_lakh"] -
    not_approved["median_price_lakh"]
)

median_rate_difference = (
    approved["median_rate_per_sqft"] -
    not_approved["median_rate_per_sqft"]
)

print("\nObserved median differences:")
print(f"Median price difference (₹ lakh): {median_price_difference:.2f}")
print(f"Median rate difference (₹/sqft): {median_rate_difference:.2f}")

RERA approval comparison:
       rera_approval  listings  avg_price_lakh  median_price_lakh  avg_rate_per_sqft  median_rate_per_sqft  avg_area_sqft  median_area_sqft
    Approved by RERA      5450      386.039684              270.5       17643.496697               14200.0    2079.910459            1800.0
Not approved by RERA      8775      407.868709              258.0       13836.737208               11544.0    2958.131396            2200.0

RERA approval by property type:
flat_type             Apartment  Floor  House  Penthouse  Plot  Villa
rera_approval                                                        
Approved by RERA           4956    145     14          1   332      2
Not approved by RERA       5939    736     31         31  1835    203

RERA approval by BHK:
bhk_group               1     2     3     4    5   6   7  8+ BHK  Not specified
rera_approval                                                                  
Approved by RERA      106  1010  2507  1360  146  19   0  